In [ ]:
import pandas as pd

In [ ]:
vehicle = pd.read_excel("./Data Extraction Sheet.xlsx", sheet_name="Vehicle Extraction")
vehicle

In [ ]:
vehicle["Data need"].unique()

In [ ]:
hemoglobin_effect_rows = vehicle[
    vehicle["Data need"] == "Iron fortification/consumption effect on hemoglobin"
].copy()
# Two parameterizations share this data need: the per-vehicle fixed mean
# differences (historical, kept for reference) and the Dorbu dose-response
# slope the models now consume. Split by data point name.
assert set(hemoglobin_effect_rows["Data point name"]) == {
    "Mean difference",
    "Dose-response slope",
}
hemoglobin_effects = hemoglobin_effect_rows[
    hemoglobin_effect_rows["Data point name"] == "Mean difference"
].copy()
assert (hemoglobin_effects["Units"] == "g/L").all()
hemoglobin_effects = hemoglobin_effects[["Vehicle", "Value"]].rename(
    columns={"Vehicle": "vehicle_name", "Value": "value"}
)
hemoglobin_effects["vehicle_name"] = hemoglobin_effects.vehicle_name.str.lower()
hemoglobin_effects

In [ ]:
results_dir = "../results"
hemoglobin_effects.to_csv(
    f"{results_dir}/iron/fortification_hemoglobin_effects.csv", index=False
)

In [ ]:
# The dose-response slope (Dorbu et al. 2026, Matern Child Nutr 22:e13801):
# hemoglobin response per mg/day of fortificant iron, vehicle-independent.
# This is what the maternal sim and the 0400 anemia notebook consume; the
# per-vehicle mean differences above are retained for reference only.
hemoglobin_effect_per_mg = hemoglobin_effect_rows[
    hemoglobin_effect_rows["Data point name"] == "Dose-response slope"
].copy()
assert len(hemoglobin_effect_per_mg) == 1
assert (hemoglobin_effect_per_mg["Units"] == "g/L per mg/day").all()
# The 'parameter' column exists because the artifact loader indexes CSVs by
# every non-value column and cannot build an empty index.
hemoglobin_effect_per_mg = (
    hemoglobin_effect_per_mg[["Value"]]
    .rename(columns={"Value": "value"})
    .assign(parameter="per_mg_daily_intake")
)[["parameter", "value"]]
hemoglobin_effect_per_mg

In [ ]:
hemoglobin_effect_per_mg.to_csv(
    f"{results_dir}/iron/fortification_hemoglobin_effect_per_mg.csv", index=False
)

In [ ]:
birthweight_effects = vehicle[
    vehicle["Data need"] == "Iron fortification/consumption effect on birthweight"
].copy()
assert (birthweight_effects["Data point name"] == "Mean difference").all()
assert (birthweight_effects["Units"] == "g/(mg/person-day)").all()
birthweight_effects = birthweight_effects[["Vehicle", "Value"]].rename(
    columns={"Vehicle": "vehicle_name", "Value": "value"}
)
birthweight_effects["vehicle_name"] = birthweight_effects.vehicle_name.str.lower()
birthweight_effects

In [ ]:
birthweight_effects.to_csv(
    f"{results_dir}/iron/fortification_birthweight_effects.csv", index=False
)